# Softmax

回归问题通常为多个输出，输出 i 是预测为第 i 类的置信度

对类别进行编码，独热编码：独热编码是一个向量，它的分量和类别一样多，类别对应的分量设置为 1，其他所有分量设置为 0

为了确保最终输出的概率值总和为 1，我们再让每个求幂后的结果除以它们的总和：

$$\hat{\mathbf{y}} = \text{softmax}(\mathbf{o}) \quad \text{其中} \quad \hat{y}_j = \frac{\exp(o_j)}{\sum_k \exp(o_k)}$$

用交叉熵衡量两个概率的区别：

$$l(\mathbf{y}, \hat{\mathbf{y}}) = - \sum_{i} y_i \log \hat{y}_i = - \log \hat{y}_y$$

求导后：
$$\frac{\partial L}{\partial o_i} = \hat{y}_i - y_i$$

## 图像分类数据集

使用 Fashion-MNIST 

In [1]:
import torch
import torchvision
from torch.utils import data
from torchvision import transforms

# 准备数据集
trans = transforms.ToTensor()
mnist_train = torchvision.datasets.FashionMNIST(root="./data", train=True, transform=trans, download=True)
mnist_test = torchvision.datasets.FashionMNIST(root="./data", train=False, transform=trans, download=True)

# 读取小批量
batch_size = 256

def get_dataloader_workers():
    return 4

# 加载数据
train_iter = data.DataLoader(mnist_train, batch_size, shuffle=True, num_workers=get_dataloader_workers())
test_iter = data.DataLoader(mnist_test, batch_size, shuffle=False, num_workers=get_dataloader_workers())

100.0%
100.0%
100.0%
100.0%


## Softmax 从 0 实现

In [ ]:
# 图片为 784 像素，输出为 10 个类别
num_inputs = 784
num_outputs = 10

# 权重初始化为均值为 0、标准差为 0.01 的正态分布
W = torch.normal(0, 0.01, size=(num_inputs, num_outputs), requires_grad=True)
b = torch.zeros(num_outputs, requires_grad=True)

# 定义 Softmax 操作
def softmax(X):
    X_exp = torch.exp(X)
    partition = X_exp.sum(1, keepdim=True)
    return X_exp / partition

# 定义模型
def net(X):
    return softmax(torch.matmul(X.reshape((-1, W.shape[0])), W) + b)

# 交叉熵损失函数
def cross_entropy(y_hat, y):
    return -torch.log(y_hat[range(len(y_hat)), y])

# 计算分类准确率
def accuracy(y_hat, y):
    if len(y_hat.shape) > 1 and y_hat.shape[1] > 1:
        y_hat = y_hat.argmax(axis=1) # 找到概率最大的类别索引
    cmp = y_hat.type(y.dtype) == y    # 比较预测和真实标签
    return float(cmp.type(y.dtype).sum()) # 返回正确预测的个数

# 评估 net 准确率
def evaluate_accuracy(net, data_iter):
    if isinstance(net, torch.nn.Module):
        net.eval()
    metric = Accumulator(2)
    for X, y in data_iter:
        metric.add(accuracy(net(X), y), y.numel())
    return metric[0] / metric[1]

# Accumulator 实现
class Accumulator:
    def __init__(self, n):
        self.data = [0.0] * n

    def add(self, *args):
        self.data = [a + float(b) for a, b in zip(self.data, args)]

    def reset(self):
        self.data = [0.0] * len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]
    
# 定义优化器
def sgd(params, lr, batch_size):
    with torch.no_grad():
        for param in params:
            param -= lr * param.grad / batch_size
            param.grad.zero_()
# 训练
def train_epoch(net, train_iter, loss, updater):  #@save
    """训练模型一个迭代周期"""
    # 将模型设置为训练模式
    if isinstance(net, torch.nn.Module):
        net.train()
    # 训练损失总和、训练准确度总和、样本数
    metric = Accumulator(3)
    for X, y in train_iter:
        # 计算梯度并更新参数
        y_hat = net(X)
        l = loss(y_hat, y)
        if isinstance(updater, torch.optim.Optimizer):
            # 使用PyTorch内置的优化器和损失函数
            updater.zero_grad()
            l.mean().backward()
            updater.step()
        else:
            # 使用定制的优化器和损失函数
            l.sum().backward()
            updater(X.shape[0])
        metric.add(float(l.sum()), accuracy(y_hat, y), y.numel())
    # 返回训练损失和训练精度
    return metric[0] / metric[2], metric[1] / metric[2]

def train_ch3(net, train_iter, test_iter, loss, num_epochs, updater):
    for epoch in range(num_epochs):
        train_metrics = train_epoch(net, train_iter, loss, updater)
        test_acc = evaluate_accuracy(net, test_iter)
        print(f'epoch {epoch + 1}: loss {train_metrics[0]:.4f}, '
              f'train acc {train_metrics[1]:.3f}, test acc {test_acc:.3f}')

lr = 0.1
updater = lambda batch_size: sgd([W, b], lr, batch_size)
num_epochs = 10

## Pytorch 实现

In [ ]:
import torch
from torch import nn, optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# 1. 准备数据 
transform = transforms.ToTensor()
train_dataset = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

train_iter = DataLoader(train_dataset, batch_size=256, shuffle=True)
test_iter = DataLoader(test_dataset, batch_size=256, shuffle=False)

# 2. 定义模型
# nn.Flatten: 将 (256, 1, 28, 28) 转换为 (256, 784)
# nn.Linear: 自动定义 W (784, 10) 和 b (10,)
net = nn.Sequential(nn.Flatten(), nn.Linear(784, 10))

# 3. 初始化参数
def init_weights(m):
    if type(m) == nn.Linear:
        nn.init.normal_(m.weight, std=0.01)

net.apply(init_weights)

# 4. 定义损失函数与优化器
# CrossEntropyLoss 在 PyTorch 中自带了 Softmax，所以模型最后不用加 Softmax 层
loss = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.1)

# 5. 训练循环
num_epochs = 10
for epoch in range(num_epochs):
    net.train() # 切换到训练模式
    running_loss = 0.0
    correct = 0
    total = 0
    
    for X, y in train_iter:
        # 梯度清零
        optimizer.zero_grad()
        # 前向传播
        y_hat = net(X)
        l = loss(y_hat, y)
        # 反向传播与更新
        l.backward()
        optimizer.step()
        
        # 统计指标
        running_loss += l.item() * X.shape[0]
        correct += (y_hat.argmax(1) == y).sum().item()
        total += y.numel()
    
    # 在测试集上评估
    net.eval() # 切换到预测模式
    test_correct = 0
    test_total = 0
    with torch.no_grad():
        for X_test, y_test in test_iter:
            outputs = net(X_test)
            test_correct += (outputs.argmax(1) == y_test).sum().item()
            test_total += y_test.numel()
            
    print(f'Epoch {epoch + 1}: Loss {running_loss / total:.4f}, '
          f'Train Acc {correct / total * 100:.2f}%, '
          f'Test Acc {test_correct / test_total * 100:.2f}%')